In [1]:

from lume_cheetah import LUMECheetahModel, CheetahSimulator
from lume_cheetah.transformer import SLACCheetahTransformer
from cheetah.accelerator import Segment
from cheetah.particles import ParticleBeam
from lume_cheetah.model_configs.loading import variables_from_yaml
from lume_cheetah.utils import get_pv_mad_mapping
from lume_cheetah.model_configs.generation import ModelVariableGeneration
import torch
import os




#TODO: clean up example
#TODO: Move slac pv -> cheetah ele mapping into slac_utils.py
#TODO: make large modifications to model_config_generation.py

# Create Segment from lattice file of choice (should be consistent with model)
lattice_fp = os.path.join("lume_cheetah", "lattices", "nc_hxr.json")
segment = Segment.from_lattice_json( lattice_fp )

# Create incoming particle beam distribution (for beam at the start of lattice)
beam_fp = os.path.join("lume_cheetah", "beams", "impact_inj_output_YAG03.h5")
incoming_beam = ParticleBeam.from_openpmd_file(
    path= beam_fp,
    energy=torch.tensor(64e6),
    dtype=torch.float32,
)
incoming_beam.particle_charges = torch.tensor(1.0)


# Construct simulator with segment and incoming_beam_distribution
simulator = CheetahSimulator(
    segment=segment,
    initial_beam_distribution=incoming_beam,
)


# Load variables supported by the model from a yaml
# Note the burden of variables being compatible with the model is on the user
# But we can provide utilities to make it easier to generate compatible variables
#model_config_fp = os.path.join("lume_cheetah", "model_configs", "model_config_nc_injector_DL1.yaml")
#currently a problem with model generation.
#control_variables, output_variables = variables_from_yaml(model_config_fp)
#cv = { **control_variables, **output_variables}


# Or create a set of compatible variables using the Model Variable Generator
rel_areas = ['DL1.yaml', 'GUN.yaml', 'L0.yaml']
model_variable_generator = ModelVariableGeneration(rel_areas, segment)
control_variables, output_variables = model_variable_generator.variables
cv = { **control_variables, **output_variables}

#Some overlap here...

#Generate mapping from slac PV names to cheetah variable names using the mapping file
mapping_fp = os.path.join("lume_cheetah","mappings", "lcls_elements.csv")
mapping = get_pv_mad_mapping(mapping_fp)
mapped = {name: mapping[name.rsplit(":",1)[0]].lower() for name in cv if name.rsplit(":",1)[0] in mapping}
transformer = SLACCheetahTransformer(mapped)

model = LUMECheetahModel(    
simulator=simulator,
transformer=transformer,
control_variables=control_variables,
observable_variables=output_variables,
)



/sdf/home/c/cgarnier/.conda/envs/linac-simulation/lib/python3.11/site-packages/torch/nn/modules/module.py:2066: PhysicsWarning: Invalid tracking method 'cheetah' for element dl00 of type Drift, supported methods are ['linear', 'second_order', 'drift_kick_drift']. Keeping the previous tracking method linear.
  super().__setattr__(name, value)
/sdf/home/c/cgarnier/.conda/envs/linac-simulation/lib/python3.11/site-packages/torch/nn/modules/module.py:2066: PhysicsWarning: Invalid tracking method 'cheetah' for element loadlock of type Drift, supported methods are ['linear', 'second_order', 'drift_kick_drift']. Keeping the previous tracking method linear.
  super().__setattr__(name, value)
/sdf/home/c/cgarnier/.conda/envs/linac-simulation/lib/python3.11/site-packages/torch/nn/modules/module.py:2066: PhysicsWarning: Invalid tracking method 'cheetah' for element dl01a of type Drift, supported methods are ['linear', 'second_order', 'drift_kick_drift']. Keeping the previous tracking method linear

device='bpm10' dev_type='BPM' pv_attr='TMIT' pv='BPMS:IN20:581:TMIT' -> 'TMIT'
device='bpm11' dev_type='BPM' pv_attr='TMIT' pv='BPMS:IN20:631:TMIT' -> 'TMIT'
device='bpm12' dev_type='BPM' pv_attr='TMIT' pv='BPMS:IN20:651:TMIT' -> 'TMIT'
device='bpm13' dev_type='BPM' pv_attr='TMIT' pv='BPMS:IN20:731:TMIT' -> 'TMIT'
device='bpm14' dev_type='BPM' pv_attr='TMIT' pv='BPMS:IN20:771:TMIT' -> 'TMIT'
device='bpm15' dev_type='BPM' pv_attr='TMIT' pv='BPMS:IN20:781:TMIT' -> 'TMIT'
device='bpm6' dev_type='BPM' pv_attr='TMIT' pv='BPMS:IN20:425:TMIT' -> 'TMIT'
device='bpm8' dev_type='BPM' pv_attr='TMIT' pv='BPMS:IN20:511:TMIT' -> 'TMIT'
device='bpm9' dev_type='BPM' pv_attr='TMIT' pv='BPMS:IN20:525:TMIT' -> 'TMIT'
device='bpm2' dev_type='BPM' pv_attr='TMIT' pv='BPMS:IN20:221:TMIT' -> 'TMIT'
device='bpm3' dev_type='BPM' pv_attr='TMIT' pv='BPMS:IN20:235:TMIT' -> 'TMIT'


In [2]:
transformer.get_cheetah_property(simulator, 'QUAD:IN20:731:BCTRL' )

tensor(15.2427)

## Test model setting and getting

In [9]:
l = list(model.supported_variables.keys())
values_dict = {name: 1.0 for name in l if name in model.control_variables and 'BACT' not in name}
#set is very slow due to beam energy calculation in transformer, maybe some list format should be passable for args.


In [4]:
model.get(l)

{'QUAD:IN20:731:BCTRL': tensor(15.2427),
 'QUAD:IN20:425:BCTRL': tensor(-0.1907),
 'QUAD:IN20:441:BCTRL': tensor(0.5444),
 'QUAD:IN20:511:BCTRL': tensor(8.0429),
 'QUAD:IN20:525:BCTRL': tensor(-4.7208),
 'QUAD:IN20:631:BCTRL': tensor(10.4868),
 'QUAD:IN20:651:BCTRL': tensor(-8.3220),
 'QUAD:IN20:771:BCTRL': tensor(-5.7672),
 'QUAD:IN20:781:BCTRL': tensor(9.2735),
 'XCOR:IN20:521:BCTRL': tensor(0.),
 'XCOR:IN20:641:BCTRL': tensor(0.),
 'XCOR:IN20:721:BCTRL': tensor(0.),
 'XCOR:IN20:761:BCTRL': tensor(0.),
 'YCOR:IN20:522:BCTRL': tensor(0.),
 'YCOR:IN20:642:BCTRL': tensor(0.),
 'YCOR:IN20:722:BCTRL': tensor(0.),
 'YCOR:IN20:762:BCTRL': tensor(0.),
 'XCOR:IN20:221:BCTRL': tensor(0.),
 'YCOR:IN20:222:BCTRL': tensor(0.),
 'QUAD:IN20:361:BCTRL': tensor(-4.3418),
 'QUAD:IN20:371:BCTRL': tensor(4.7358),
 'BPMS:IN20:581:X': tensor(-3.2940e-09),
 'BPMS:IN20:581:Y': tensor(1.1937e-09),
 'BPMS:IN20:631:X': tensor(-8.7554e-09),
 'BPMS:IN20:631:Y': tensor(1.7039e-09),
 'BPMS:IN20:651:X': tensor(-2.4

In [ ]:
model.set(values_dict)
model.get(l)

{'QUAD:IN20:731:BCTRL': tensor(1.),
 'QUAD:IN20:425:BCTRL': tensor(1.),
 'QUAD:IN20:441:BCTRL': tensor(1.),
 'QUAD:IN20:511:BCTRL': tensor(1.),
 'QUAD:IN20:525:BCTRL': tensor(1.),
 'QUAD:IN20:631:BCTRL': tensor(1.),
 'QUAD:IN20:651:BCTRL': tensor(1.),
 'QUAD:IN20:771:BCTRL': tensor(1.),
 'QUAD:IN20:781:BCTRL': tensor(1.),
 'XCOR:IN20:521:BCTRL': tensor(1.),
 'XCOR:IN20:641:BCTRL': tensor(1.),
 'XCOR:IN20:721:BCTRL': tensor(1.),
 'XCOR:IN20:761:BCTRL': tensor(1.),
 'YCOR:IN20:522:BCTRL': tensor(1.),
 'YCOR:IN20:642:BCTRL': tensor(1.),
 'YCOR:IN20:722:BCTRL': tensor(1.),
 'YCOR:IN20:762:BCTRL': tensor(1.),
 'XCOR:IN20:221:BCTRL': tensor(1.),
 'YCOR:IN20:222:BCTRL': tensor(1.),
 'QUAD:IN20:361:BCTRL': tensor(1.0000),
 'QUAD:IN20:371:BCTRL': tensor(1.0000),
 'BPMS:IN20:581:X': tensor(-0.5781),
 'BPMS:IN20:581:Y': tensor(9.1328),
 'BPMS:IN20:631:X': tensor(-0.3556),
 'BPMS:IN20:631:Y': tensor(12.0599),
 'BPMS:IN20:651:X': tensor(-0.1982),
 'BPMS:IN20:651:Y': tensor(14.0496),
 'BPMS:IN20:731

In [13]:
model.reset()

In [ ]:
model.get(l)